# Altair で学ぶ対話的可視化 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
Python の可視化ライブラリ **Altair** の基礎を学ぶためのチュートリアルです。
Altair は「何を・どう対応づけて描くか」を宣言するだけで、ズーム・ツールチップ・選択などの
**対話機能付きのグラフ** をブラウザ上に描けます。

## 対象者
- pandas の DataFrame の基本を理解している方
- matplotlib でグラフを描いたことがあり、対話的なグラフやダッシュボード風の可視化に進みたい方
- 経済データ（地域別・年次のパネルデータなど）を見やすく伝えたい方

## このチュートリアルで学ぶこと
1. Altair とは（宣言的な文法：data / mark / encoding）
2. 基本のグラフ（散布図・折れ線・棒・ヒストグラム・箱ひげ図）
3. エンコーディングの詳細（色・サイズ・形・ツールチップ・軸の設定）
4. 集計と変換（aggregate / calculate / filter）
5. 複合グラフ（layer / concat / facet）
6. 対話機能（ズーム、選択によるハイライト、連動フィルタ、ドロップダウン）
7. 経済データの分析例
8. 保存と共有

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- グラフはブラウザ側で描画されるため、マウスを乗せると値が表示され、対話操作ができます。
- 各章の最後に **練習問題** があります。解答欄に自分で書いてから、折りたたみの解答例を開いて確認しましょう。

---
## 0. 環境準備（JupyterLite 用）

Altair は JupyterLite（Pyodide）に同梱されています。念のためインストールのセルを実行してから import します。

In [ ]:
# JupyterLite 用のパッケージインストール
try:
    import piplite
    await piplite.install(["altair", "pandas", "numpy"])
except ImportError:
    pass

In [ ]:
import altair as alt
import numpy as np
import pandas as pd

print("Altair バージョン:", alt.__version__)
print("pandas バージョン:", pd.__version__)

### 使用するデータ

このノートブックでは、乱数で作った **架空の地域経済パネルデータ**（8 地域 × 2015〜2024 年）と、
**学生の学習時間と成績のデータ**（200 人）を使います。実データではありませんが、
形式は実際の統計データと同じなので、そのまま応用できます。

In [ ]:
rng = np.random.default_rng(42)

regions = ["北海道", "東北", "関東", "中部", "近畿", "中国", "四国", "九州"]
base_gdp = {"北海道": 20, "東北": 34, "関東": 210, "中部": 90, "近畿": 85, "中国": 30, "四国": 14, "九州": 48}   # 兆円
base_pop = {"北海道": 520, "東北": 850, "関東": 4400, "中部": 2100, "近畿": 2050, "中国": 720, "四国": 360, "九州": 1280}  # 万人

rows = []
for region in regions:
    gdp = base_gdp[region]
    for year in range(2015, 2025):
        growth = rng.normal(1.0, 1.5)                          # 成長率（%）
        gdp = gdp * (1 + growth / 100)
        rows.append({
            "region": region,
            "year": year,
            "gdp": round(gdp, 2),                               # 域内総生産（兆円）
            "growth": round(growth, 2),
            "unemployment": round(rng.normal(2.8, 0.5), 2),     # 失業率（%）
            "population": round(base_pop[region] * (1 - 0.003 * (year - 2015)) + rng.normal(0, 5), 1),
        })
econ = pd.DataFrame(rows)
print(econ.shape)
print(econ.head())

In [ ]:
faculties = ["経済", "法", "文", "理工"]
n = 200
students = pd.DataFrame({
    "faculty": rng.choice(faculties, size=n),
    "study_hours": rng.gamma(shape=3, scale=1.2, size=n).round(1),   # 1 日の学習時間
    "part_time": rng.choice(["あり", "なし"], size=n, p=[0.6, 0.4]),
})
students["score"] = (50 + 6 * students["study_hours"] + rng.normal(0, 8, size=n)).clip(0, 100).round(1)
print(students.head())
print(students.describe().round(2))

---
## 1. Altair とは

Altair は、統計可視化の文法 **Vega-Lite** を Python から使うためのライブラリです。グラフは次の 3 つの要素で宣言します。

| 要素 | 意味 | 例 |
|---|---|---|
| **data** | 描くデータ（DataFrame） | `alt.Chart(df)` |
| **mark** | 図形の種類 | `.mark_point()`, `.mark_line()`, `.mark_bar()` |
| **encoding** | 列と視覚属性の対応づけ | `.encode(x="col", y="col2", color="col3")` |

「どう描くか（ループで点を打つ等）」ではなく「何を何に対応づけるか」を書くのが **宣言的** な書き方です。

In [ ]:
# 最初のグラフ：学習時間と成績の散布図（マウスを乗せると値が表示される）
chart = alt.Chart(students).mark_point().encode(
    x="study_hours",
    y="score",
    tooltip=["faculty", "study_hours", "score"],
)
chart

### データ型の指定

列名の後ろに `:Q` などを付けて **データ型** を明示できます。型によって軸やスケールの扱いが変わります。

| 記号 | 型 | 例 |
|:---:|---|---|
| `:Q` | 量的（quantitative） | 売上、GDP、成績 |
| `:N` | 名義（nominal） | 地域名、学部名 |
| `:O` | 順序（ordinal） | 年（順序として扱う）、評価ランク |
| `:T` | 時間（temporal） | 日付・時刻 |

省略すると DataFrame の dtype から推定されます（数値 → Q、文字列 → N、日付 → T）。
`year` のような整数列は、そのままだと Q（連続値）として扱われるので、`:O` を付けると離散的な軸になります。

In [ ]:
# 型を明示した例：year を順序（O）として棒グラフに
kanto = econ[econ["region"] == "関東"]
alt.Chart(kanto).mark_bar().encode(
    x="year:O",
    y="gdp:Q",
    tooltip=["year", "gdp"],
).properties(title="関東の域内総生産（兆円）", width=400)

In [ ]:
# 同じ列を Q（量的）として扱うと連続的な軸になる
alt.Chart(kanto).mark_line(point=True).encode(
    x="year:Q",
    y="gdp:Q",
).properties(title="year を量的（Q）として描いた例", width=400)

### 練習問題 1

1. `students` を使って、横軸 `study_hours`、縦軸 `score`、ツールチップに `part_time` を表示する散布図を描いてください。
2. `econ` から「近畿」の行だけを取り出し、横軸 `year:O`、縦軸 `unemployment:Q` の棒グラフを描いてください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
alt.Chart(students).mark_point().encode(x="study_hours", y="score", tooltip=["part_time"])

# 2
kinki = econ[econ["region"] == "近畿"]
alt.Chart(kinki).mark_bar().encode(x="year:O", y="unemployment:Q").properties(width=400)
```

</details>

---
## 2. 基本のグラフ

mark を変えるだけで、同じ書き方でさまざまなグラフが描けます。

### 2.1 折れ線グラフ：`mark_line`

地域ごとに線を分けるには、`color` に地域名を対応づけます。

In [ ]:
alt.Chart(econ).mark_line(point=True).encode(
    x="year:O",
    y="gdp:Q",
    color="region:N",
    tooltip=["region", "year", "gdp"],
).properties(title="地域別の域内総生産の推移", width=500, height=300)

### 2.2 棒グラフ：`mark_bar`

`sort="-y"` で値の降順に並べ替えられます。

In [ ]:
latest = econ[econ["year"] == 2024]
alt.Chart(latest).mark_bar().encode(
    x=alt.X("region:N", sort="-y", title="地域"),
    y=alt.Y("gdp:Q", title="域内総生産（兆円）"),
    tooltip=["region", "gdp"],
).properties(title="2024 年の地域別 GDP", width=400)

### 2.3 ヒストグラム：`mark_bar` + `bin`

量的な列を `bin=True` で区間に分け、`count()` で件数を集計します。

In [ ]:
alt.Chart(students).mark_bar().encode(
    x=alt.X("score:Q", bin=alt.Bin(maxbins=20), title="成績"),
    y=alt.Y("count()", title="人数"),
).properties(title="成績の分布", width=400)

### 2.4 箱ひげ図：`mark_boxplot`

In [ ]:
alt.Chart(students).mark_boxplot(extent="min-max").encode(
    x=alt.X("faculty:N", title="学部"),
    y=alt.Y("score:Q", title="成績"),
    color="faculty:N",
).properties(title="学部別の成績分布", width=300)

### 2.5 面グラフ：`mark_area`

`stack` により地域を積み上げて、合計の推移と内訳を同時に見せられます。

In [ ]:
alt.Chart(econ).mark_area().encode(
    x="year:O",
    y=alt.Y("gdp:Q", stack=True, title="域内総生産（兆円）"),
    color="region:N",
    tooltip=["region", "year", "gdp"],
).properties(title="地域別 GDP の積み上げ", width=500, height=300)

### 練習問題 2

1. `econ` を使って、地域別の失業率（`unemployment`）の推移を折れ線グラフで描いてください。
2. `students` の `study_hours` のヒストグラム（区間の数は最大 15）を描いてください。
3. `students` について、`part_time`（アルバイトの有無）別の `score` の箱ひげ図を描いてください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
alt.Chart(econ).mark_line().encode(x="year:O", y="unemployment:Q", color="region:N").properties(width=500)

# 2
alt.Chart(students).mark_bar().encode(x=alt.X("study_hours:Q", bin=alt.Bin(maxbins=15)), y="count()")

# 3
alt.Chart(students).mark_boxplot().encode(x="part_time:N", y="score:Q", color="part_time:N")
```

</details>

---
## 3. エンコーディングの詳細

`encode()` には x・y のほかに、色（`color`）、大きさ（`size`）、形（`shape`）、透明度（`opacity`）、
ツールチップ（`tooltip`）などを指定できます。`alt.X()`, `alt.Color()` のようなクラスを使うと、
タイトル・書式・スケールなどを細かく設定できます。

In [ ]:
# 色（名義）と大きさ（量的）を同時に使う
alt.Chart(latest).mark_circle().encode(
    x=alt.X("gdp:Q", title="域内総生産（兆円）"),
    y=alt.Y("unemployment:Q", title="失業率（%）", scale=alt.Scale(zero=False)),
    size=alt.Size("population:Q", title="人口（万人）"),
    color=alt.Color("region:N", title="地域"),
    tooltip=["region", "gdp", "unemployment", "population"],
).properties(title="2024 年：GDP・失業率・人口", width=450, height=300)

In [ ]:
# 量的な列を色にすると、連続的なカラースケールになる（scheme で配色を変更）
alt.Chart(latest).mark_bar().encode(
    x=alt.X("region:N", sort="-y", title="地域"),
    y=alt.Y("gdp:Q", title="域内総生産（兆円）"),
    color=alt.Color("growth:Q", scale=alt.Scale(scheme="redblue", domainMid=0), title="成長率（%）"),
    tooltip=["region", "gdp", "growth"],
).properties(width=400)

In [ ]:
# 形（shape）と透明度（opacity）
alt.Chart(students).mark_point(filled=True, size=60).encode(
    x="study_hours:Q",
    y="score:Q",
    shape=alt.Shape("part_time:N", title="アルバイト"),
    color=alt.Color("faculty:N", title="学部"),
    opacity=alt.value(0.6),
).properties(width=450, height=300)

In [ ]:
# 軸の書式：format で数値の表示形式、labelAngle でラベルの角度
alt.Chart(econ[econ["region"] == "関東"]).mark_bar().encode(
    x=alt.X("year:O", title="年", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("population:Q", title="人口（万人）", axis=alt.Axis(format=",")),
).properties(title="関東の人口", width=400)

### 練習問題 3

1. `latest`（2024 年）を使い、横軸 `population`、縦軸 `gdp`、色 `region`、点の大きさ `growth` の散布図を描いてください。軸には日本語タイトルを付けます。
2. `students` の散布図で、`score` を色（量的、カラースキーム `"viridis"`）にしてください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
alt.Chart(latest).mark_circle().encode(
    x=alt.X("population:Q", title="人口（万人）"),
    y=alt.Y("gdp:Q", title="域内総生産（兆円）"),
    color=alt.Color("region:N", title="地域"),
    size=alt.Size("growth:Q", title="成長率（%）"),
    tooltip=["region", "population", "gdp", "growth"],
).properties(width=450, height=300)

# 2
alt.Chart(students).mark_circle().encode(
    x="study_hours:Q", y="score:Q",
    color=alt.Color("score:Q", scale=alt.Scale(scheme="viridis")),
)
```

</details>

---
## 4. 集計と変換

pandas で集計してから描くこともできますが、Altair には **グラフ側で集計・変換する** 仕組みもあります。
データを渡してから「どう集計するか」を宣言するので、対話機能（第 6 章）と組み合わせやすいのが利点です。

### 4.1 エンコーディング内の集計

`"mean(score):Q"` のように、集計関数を列名に付けて書けます（`mean`, `sum`, `count`, `median`, `max` など）。

In [ ]:
alt.Chart(students).mark_bar().encode(
    x=alt.X("faculty:N", title="学部"),
    y=alt.Y("mean(score):Q", title="平均点"),
    tooltip=[alt.Tooltip("mean(score):Q", format=".1f", title="平均点"), "count()"],
).properties(title="学部別の平均点", width=300)

In [ ]:
# 誤差付きの平均：層（layer）で点と誤差線を重ねる（第 5 章で詳しく）
bars = alt.Chart(students).mark_bar(opacity=0.5).encode(x="faculty:N", y="mean(score):Q")
errs = alt.Chart(students).mark_errorbar(extent="ci").encode(x="faculty:N", y=alt.Y("score:Q", title="成績"))
(bars + errs).properties(title="平均点と 95% 信頼区間", width=300)

### 4.2 transform_aggregate / transform_calculate / transform_filter

- `transform_aggregate`：集計した列を作る（`groupby` でグループ化）
- `transform_calculate`：式で新しい列を作る（式の中では `datum.列名` でデータを参照）
- `transform_filter`：条件で行を絞り込む

In [ ]:
# 地域ごとの平均成長率を集計し、降順の棒グラフに
alt.Chart(econ).transform_aggregate(
    mean_growth="mean(growth)",
    groupby=["region"],
).mark_bar().encode(
    x=alt.X("region:N", sort="-y", title="地域"),
    y=alt.Y("mean_growth:Q", title="平均成長率（%）"),
    tooltip=[alt.Tooltip("mean_growth:Q", format=".2f")],
).properties(title="2015〜2024 年の平均成長率", width=400)

In [ ]:
# 1 人あたり GDP（万円）を計算して描く：兆円 / 万人 × 1e4
alt.Chart(latest).transform_calculate(
    gdp_per_capita="datum.gdp / datum.population * 10000",
).mark_bar().encode(
    x=alt.X("region:N", sort="-y", title="地域"),
    y=alt.Y("gdp_per_capita:Q", title="1 人あたり GDP（万円）"),
    tooltip=[alt.Tooltip("gdp_per_capita:Q", format=".0f")],
).properties(title="2024 年の 1 人あたり GDP", width=400)

In [ ]:
# 条件で絞り込む：2020 年以降、成長率がマイナスだった地域・年
alt.Chart(econ).transform_filter(
    (alt.datum.year >= 2020) & (alt.datum.growth < 0)
).mark_circle(size=100, color="crimson").encode(
    x="year:O",
    y=alt.Y("region:N", title="地域"),
    tooltip=["region", "year", "growth"],
).properties(title="2020 年以降にマイナス成長だった地域・年", width=400)

### 練習問題 4

1. `students` について、学部別・アルバイト有無別の平均点を、横軸 `faculty`、色 `part_time` の棒グラフ（`xOffset="part_time:N"` で横に並べる）で描いてください。
2. `econ` に `transform_calculate` で「失業者数の概算（万人） = population × unemployment / 100」を作り、2024 年について地域別の棒グラフにしてください（`transform_filter` で 2024 年に絞ります）。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
alt.Chart(students).mark_bar().encode(
    x="faculty:N", xOffset="part_time:N", y="mean(score):Q", color="part_time:N",
).properties(width=300)

# 2
alt.Chart(econ).transform_filter(alt.datum.year == 2024).transform_calculate(
    unemployed="datum.population * datum.unemployment / 100"
).mark_bar().encode(
    x=alt.X("region:N", sort="-y"), y=alt.Y("unemployed:Q", title="失業者数（万人）"),
    tooltip=[alt.Tooltip("unemployed:Q", format=".1f")],
).properties(width=400)
```

</details>

---
## 5. 複合グラフ

複数のグラフを組み合わせる演算子が用意されています。

| 演算 | 意味 |
|---|---|
| `a + b`（`alt.layer`） | 重ねる（同じ軸） |
| `a | b`（`alt.hconcat`） | 横に並べる |
| `a & b`（`alt.vconcat`） | 縦に並べる |
| `.facet()` / `row`・`column` エンコーディング | カテゴリごとに小さなグラフを並べる（small multiples） |

In [ ]:
# layer：棒グラフの上に数値ラベルを重ねる
base = alt.Chart(latest).encode(x=alt.X("region:N", sort="-y", title="地域"))
bars = base.mark_bar().encode(y=alt.Y("gdp:Q", title="域内総生産（兆円）"))
labels = base.mark_text(dy=-6, fontSize=11).encode(y="gdp:Q", text=alt.Text("gdp:Q", format=".0f"))
(bars + labels).properties(title="2024 年の地域別 GDP", width=400)

In [ ]:
# layer：折れ線と点、平均の水平線を重ねる
kanto = econ[econ["region"] == "関東"]
line = alt.Chart(kanto).mark_line(point=True).encode(x="year:O", y=alt.Y("growth:Q", title="成長率（%）"))
rule = alt.Chart(kanto).mark_rule(color="red", strokeDash=[4, 4]).encode(y="mean(growth):Q")
(line + rule).properties(title="関東の成長率と期間平均（赤の破線）", width=400)

In [ ]:
# hconcat / vconcat：異なるグラフを並べる
scatter = alt.Chart(students).mark_point().encode(x="study_hours:Q", y="score:Q", color="faculty:N").properties(width=250, height=200)
hist = alt.Chart(students).mark_bar().encode(x=alt.X("score:Q", bin=True), y="count()").properties(width=250, height=200)
scatter | hist

In [ ]:
# facet：地域ごとに小さなグラフを並べる（small multiples）
alt.Chart(econ).mark_line().encode(
    x="year:O",
    y=alt.Y("unemployment:Q", title="失業率（%）"),
).properties(width=120, height=100).facet(
    facet=alt.Facet("region:N", title="地域"),
    columns=4,
).properties(title="地域別の失業率の推移")

In [ ]:
# row / column エンコーディングでも同じことができる
alt.Chart(students).mark_point().encode(
    x="study_hours:Q",
    y="score:Q",
    column=alt.Column("faculty:N", title="学部"),
    color="part_time:N",
).properties(width=140, height=140)

### 練習問題 5

1. `latest` の失業率の棒グラフの上に、値（小数 1 桁）のテキストラベルを重ねてください。
2. `econ` について、地域ごとの GDP の推移を `facet`（4 列）で並べてください。
3. 「地域別 GDP の棒グラフ」と「地域別失業率の棒グラフ」を横に並べてください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
base = alt.Chart(latest).encode(x=alt.X("region:N", sort="-y"))
bars = base.mark_bar().encode(y=alt.Y("unemployment:Q", title="失業率（%）"))
text = base.mark_text(dy=-6).encode(y="unemployment:Q", text=alt.Text("unemployment:Q", format=".1f"))
(bars + text).properties(width=400)

# 2
alt.Chart(econ).mark_line().encode(x="year:O", y="gdp:Q").properties(width=120, height=100).facet("region:N", columns=4)

# 3
g = alt.Chart(latest).mark_bar().encode(x=alt.X("region:N", sort="-y"), y="gdp:Q").properties(width=250)
u = alt.Chart(latest).mark_bar(color="orange").encode(x=alt.X("region:N", sort="-y"), y="unemployment:Q").properties(width=250)
g | u
```

</details>

---
## 6. 対話機能

Altair の大きな特徴は、コードを少し足すだけで対話機能が付くことです。

- `.interactive()`：マウスのドラッグとホイールでパン・ズーム
- `alt.selection_point()`：クリックで要素を選択（凡例に結び付けることもできる）
- `alt.selection_interval()`：ドラッグで範囲を選択（ブラシ）
- `alt.condition(選択, 選択時の値, 非選択時の値)`：選択状態に応じて見た目を変える
- `bind=alt.binding_select(...)`：ドロップダウンなどの UI 部品と結び付ける

選択（パラメータ）は `.add_params()` でグラフに登録します。

In [ ]:
# ズーム・パン
alt.Chart(students).mark_point().encode(
    x="study_hours:Q", y="score:Q", color="faculty:N", tooltip=["faculty", "study_hours", "score"],
).properties(width=450, height=300, title="ドラッグで移動、ホイールで拡大縮小").interactive()

In [ ]:
# 凡例をクリックすると、その地域だけがハイライトされる
select_region = alt.selection_point(fields=["region"], bind="legend")

alt.Chart(econ).mark_line(point=True).encode(
    x="year:O",
    y=alt.Y("gdp:Q", title="域内総生産（兆円）"),
    color="region:N",
    opacity=alt.condition(select_region, alt.value(1.0), alt.value(0.15)),
    tooltip=["region", "year", "gdp"],
).add_params(select_region).properties(width=500, height=300, title="凡例をクリック（Shift で複数選択）")

In [ ]:
# 範囲選択（ブラシ）で 2 つのグラフを連動させる
brush = alt.selection_interval()

points = alt.Chart(students).mark_point().encode(
    x="study_hours:Q",
    y="score:Q",
    color=alt.condition(brush, "faculty:N", alt.value("lightgray")),
).add_params(brush).properties(width=350, height=250, title="ドラッグで範囲を選択")

bars = alt.Chart(students).mark_bar().encode(
    x=alt.X("faculty:N", title="学部"),
    y=alt.Y("count()", title="選択された人数"),
    color="faculty:N",
).transform_filter(brush).properties(width=200, height=250)

points | bars

In [ ]:
# ドロップダウンで地域を選ぶ
dropdown = alt.binding_select(options=regions, name="地域: ")
select_one = alt.selection_point(fields=["region"], bind=dropdown, value="関東")

alt.Chart(econ).mark_bar().encode(
    x="year:O",
    y=alt.Y("growth:Q", title="成長率（%）"),
    color=alt.condition(alt.datum.growth > 0, alt.value("steelblue"), alt.value("crimson")),
    tooltip=["region", "year", "growth"],
).add_params(select_one).transform_filter(select_one).properties(width=400, title="選んだ地域の成長率")

In [ ]:
# スライダーで年を選び、その年の横断面を見る
slider = alt.binding_range(min=2015, max=2024, step=1, name="年: ")
select_year = alt.selection_point(fields=["year"], bind=slider, value=2024)

alt.Chart(econ).mark_circle(size=120).encode(
    x=alt.X("gdp:Q", title="域内総生産（兆円）", scale=alt.Scale(domain=[0, 260])),
    y=alt.Y("unemployment:Q", title="失業率（%）", scale=alt.Scale(domain=[1, 5])),
    color="region:N",
    tooltip=["region", "year", "gdp", "unemployment"],
).add_params(select_year).transform_filter(select_year).properties(width=450, height=300, title="スライダーで年を変更")

### 練習問題 6

1. `students` の散布図に `.interactive()` を付け、ツールチップに全列を表示してください。
2. `econ` の失業率の折れ線グラフに、凡例クリックで地域をハイライトする選択を付けてください。
3. ブラシ（`selection_interval`）で選んだ範囲の学生について、`part_time` 別の人数を棒グラフで表示する連動グラフを作ってください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
alt.Chart(students).mark_point().encode(
    x="study_hours:Q", y="score:Q", color="faculty:N", tooltip=list(students.columns),
).interactive()

# 2
sel = alt.selection_point(fields=["region"], bind="legend")
alt.Chart(econ).mark_line().encode(
    x="year:O", y="unemployment:Q", color="region:N",
    opacity=alt.condition(sel, alt.value(1), alt.value(0.15)),
).add_params(sel).properties(width=500)

# 3
brush = alt.selection_interval()
pts = alt.Chart(students).mark_point().encode(
    x="study_hours:Q", y="score:Q", color=alt.condition(brush, "part_time:N", alt.value("lightgray")),
).add_params(brush).properties(width=300)
cnt = alt.Chart(students).mark_bar().encode(x="part_time:N", y="count()", color="part_time:N").transform_filter(brush)
pts | cnt
```

</details>

---
## 7. 経済データの分析例

これまでの機能を組み合わせて、パネルデータを多面的に見てみます。

In [ ]:
# 例 1：成長率と失業率の関係（回帰直線を重ねる）
base = alt.Chart(econ).encode(
    x=alt.X("growth:Q", title="成長率（%）"),
    y=alt.Y("unemployment:Q", title="失業率（%）", scale=alt.Scale(zero=False)),
)
points = base.mark_circle(opacity=0.5).encode(color="region:N", tooltip=["region", "year", "growth", "unemployment"])
trend = base.transform_regression("growth", "unemployment").mark_line(color="black")
(points + trend).properties(width=450, height=300, title="成長率と失業率（全地域・全年）")

In [ ]:
# 例 2：2015 年を 100 とした GDP の指数（地域間の成長を比較）
first = econ[econ["year"] == 2015][["region", "gdp"]].rename(columns={"gdp": "gdp_2015"})
indexed = econ.merge(first, on="region")
indexed["index"] = (indexed["gdp"] / indexed["gdp_2015"] * 100).round(1)

sel = alt.selection_point(fields=["region"], bind="legend")
alt.Chart(indexed).mark_line(point=True).encode(
    x="year:O",
    y=alt.Y("index:Q", title="GDP 指数（2015 年 = 100）", scale=alt.Scale(zero=False)),
    color="region:N",
    opacity=alt.condition(sel, alt.value(1), alt.value(0.15)),
    tooltip=["region", "year", "index"],
).add_params(sel).properties(width=500, height=300, title="地域別 GDP 指数")

In [ ]:
# 例 3：ランキングの推移（ヒートマップ）
alt.Chart(econ).transform_window(
    rank="rank(gdp)",
    sort=[alt.SortField("gdp", order="descending")],
    groupby=["year"],
).mark_rect().encode(
    x="year:O",
    y=alt.Y("region:N", title="地域"),
    color=alt.Color("rank:O", scale=alt.Scale(scheme="blues", reverse=True), title="順位"),
    tooltip=["region", "year", "gdp", "rank:O"],
).properties(width=400, title="GDP 順位の推移（濃いほど上位）")

### 練習問題 7

1. 2024 年の横断面で、横軸 1 人あたり GDP（`gdp / population * 10000`）、縦軸失業率の散布図に地域名のテキストラベルを重ねてください（`mark_text(dx=8, align="left")`）。
2. `indexed` を使って、2024 年の GDP 指数が高い順に地域を並べた棒グラフを描いてください。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```python
# 1
base = alt.Chart(latest).transform_calculate(gpc="datum.gdp / datum.population * 10000").encode(
    x=alt.X("gpc:Q", title="1 人あたり GDP（万円）"), y=alt.Y("unemployment:Q", title="失業率（%）", scale=alt.Scale(zero=False)),
)
(base.mark_circle(size=100) + base.mark_text(dx=8, align="left").encode(text="region:N")).properties(width=450, height=300)

# 2
alt.Chart(indexed[indexed["year"] == 2024]).mark_bar().encode(
    x=alt.X("region:N", sort="-y"), y=alt.Y("index:Q", title="GDP 指数（2015=100）"),
).properties(width=400)
```

</details>

---
## 8. 保存と共有

グラフは `save()` で **HTML ファイル** として保存できます（対話機能もそのまま残ります）。
保存したファイルは左のファイルブラウザに現れ、右クリック → Download で手元に保存できます。
PNG などの画像への変換には追加ライブラリ（`vl-convert`）が必要で、JupyterLite では使えません。
その場合は、グラフ右上の「…」メニューから **Save as PNG / SVG** を選んでください。

In [ ]:
chart = alt.Chart(latest).mark_bar().encode(
    x=alt.X("region:N", sort="-y", title="地域"),
    y=alt.Y("gdp:Q", title="域内総生産（兆円）"),
    tooltip=["region", "gdp"],
).properties(title="2024 年の地域別 GDP", width=400)

chart.save("gdp_2024.html")
print("gdp_2024.html を保存しました")

# グラフの定義は JSON（Vega-Lite 仕様）として取り出せる
spec = chart.to_json()
print(spec[:300], "...")

---
## まとめ

| トピック | 主な書き方 |
|---|---|
| 基本 | `alt.Chart(df).mark_*().encode(x=, y=, color=, tooltip=)` |
| データ型 | `列:Q`（量的）, `:N`（名義）, `:O`（順序）, `:T`（時間） |
| mark | `mark_point`, `mark_circle`, `mark_line`, `mark_bar`, `mark_area`, `mark_boxplot`, `mark_text`, `mark_rule`, `mark_rect` |
| 軸・凡例 | `alt.X(..., title=, axis=alt.Axis(format=))`, `alt.Scale(zero=False, scheme=)`, `sort="-y"` |
| 集計・変換 | `"mean(col):Q"`, `bin=`, `transform_aggregate`, `transform_calculate`, `transform_filter`, `transform_window`, `transform_regression` |
| 複合 | `a + b`, `a | b`, `a & b`, `.facet(...)`, `column=` |
| 対話 | `.interactive()`, `selection_point`, `selection_interval`, `alt.condition`, `binding_select`, `binding_range`, `.add_params()`, `.transform_filter(sel)` |
| 保存 | `chart.save("x.html")`, `chart.to_json()` |

## 次のステップ
- `python/plotly/plotly_beginner_tutorial.ipynb` — もう一つの対話的可視化ライブラリ Plotly
- `python/seaborn/seaborn_beginner_tutorial.ipynb` — 統計的な可視化（静的）
- `python/pandas/pandas_intermediate_tutorial.ipynb` — グラフの前処理（groupby・結合・時系列）

---
## 総合演習：地域経済ダッシュボード

`econ` を使って、次の要件を満たす **1 枚の複合グラフ** を作ってください。

1. 上段：地域別 GDP の推移の折れ線グラフ。凡例クリックで地域をハイライトできる。
2. 下段左：スライダーで選んだ年の、地域別失業率の棒グラフ（降順）。
3. 下段右：同じ年の、GDP（横軸）と失業率（縦軸）の散布図。点の大きさは人口、ツールチップに地域名と値。
4. 上段と下段を `&`（縦）と `|`（横）で組み合わせ、全体にタイトルを付ける。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

In [ ]:
sel_region = alt.selection_point(fields=["region"], bind="legend")
slider = alt.binding_range(min=2015, max=2024, step=1, name="年: ")
sel_year = alt.selection_point(fields=["year"], bind=slider, value=2024)

top = alt.Chart(econ).mark_line(point=True).encode(
    x="year:O",
    y=alt.Y("gdp:Q", title="域内総生産（兆円）"),
    color=alt.Color("region:N", title="地域"),
    opacity=alt.condition(sel_region, alt.value(1), alt.value(0.15)),
    tooltip=["region", "year", "gdp"],
).add_params(sel_region).properties(width=560, height=220, title="地域別 GDP の推移（凡例クリックでハイライト）")

bars = alt.Chart(econ).mark_bar().encode(
    x=alt.X("region:N", sort="-y", title="地域"),
    y=alt.Y("unemployment:Q", title="失業率（%）"),
    color="region:N",
    tooltip=["region", "unemployment"],
).add_params(sel_year).transform_filter(sel_year).properties(width=260, height=220, title="選んだ年の失業率")

scatter = alt.Chart(econ).mark_circle().encode(
    x=alt.X("gdp:Q", title="域内総生産（兆円）", scale=alt.Scale(domain=[0, 260])),
    y=alt.Y("unemployment:Q", title="失業率（%）", scale=alt.Scale(domain=[1, 5])),
    size=alt.Size("population:Q", title="人口（万人）"),
    color="region:N",
    tooltip=["region", "gdp", "unemployment", "population"],
).transform_filter(sel_year).properties(width=260, height=220, title="GDP と失業率")

(top & (bars | scatter)).properties(title="地域経済ダッシュボード（架空データ）")

お疲れさまでした！ Altair の「宣言して描く」スタイルに慣れると、集計・複合・対話を数行ずつ足していくだけで
分析用のダッシュボードが作れます。自分のデータ（CSV を読み込んだ DataFrame）で同じ手順を試してみてください。